# Track 04b (심화) — pgvector & 검색 전략

### pgvector & 검색 전략이란?

04a에서는 Postgres 없이 메모리 안에서 RAG를 만들어 봤습니다. 운영 환경의 RAG는 **임베딩을 영구 저장**하고 **빠르게 최근접 검색**해야 하므로, Postgres 확장인 **pgvector**에 벡터를 인덱싱합니다. 이 노트북에서는 (1) pgvector 테이블에 **데이터가 제대로 적재됐는지**(행 수와 저장 차원이 `.env` 설정과 일치하는지) 점검하고, (2) 같은 질의를 **vector·graph·hybrid** 세 전략으로 실행해 검색 방식의 차이를 비교합니다. 운영 RAG 디버깅은 대개 "모델"보다 "**데이터가 제대로 적재되고 검색되는가**"를 확인하는 일에서 시작합니다.

### 이 노트북에서 보여줄 것

| Session | 무엇을 배우나요 / 왜 중요한가요 |
|---|---|
| **1. pgvector 셋업 & 인덱싱 점검** | Postgres와 임베딩 서버가 준비됐는지 먼저 확인한 뒤, 벡터 테이블의 행 수와 저장 차원(`dim_match`)을 점검합니다. 마지막으로 어댑터를 통해 샘플 질의를 검색해 *데이터가 실제로 적재되고 읽히는지부터* 확인하는 운영 습관을 익힙니다. |
| **2. 검색 전략 비교** | 10개 질의를 vector(코사인)·graph(키워드/관계)·hybrid(RRF 결합)로 실행해 검색 결과 수와 점수를 질의별로 나란히 비교합니다. 무관 질의(q10)에서 전략별로 "검색 결과 없음"을 어떻게 판정해야 하는지도 확인합니다. |

### 이 노트북을 마치면

- pgvector 테이블의 **행 수와 저장 차원**을 점검하고, `.env`에 적힌 차원과 실제 저장 차원이 어긋나는 **불일치(`dim_match`)** 문제를 찾을 수 있습니다.
- `VectorRetrievalStrategy`·`GraphRetrievalStrategy`·`HybridRetrievalStrategy`를 **같은 질의로 비교**하고, 세 전략의 점수 척도가 왜 다른지(코사인/키워드/RRF)를 설명할 수 있습니다.
- 무관한 질의에서도 **ANN 검색이 임계값 없이 낮은 점수의 결과를 반환**할 수 있음을 이해하고, 임계값·재랭킹 같은 대응을 떠올릴 수 있습니다.

> **이 노트북은 전체 인프라를 전제로 합니다.** Postgres+pgvector·TEI 임베딩 서버·graph 테이블이 없으면 거의 모든 셀이 `[SKIP]` 안내만 출력합니다. 정상 동작입니다. 실제 비교 결과를 보려면 `infrastructure/setup`의 step1~4(Postgres·MS MARCO 적재·graph)를 먼저 실행하세요. 04a 도입부의 "(선택) 임베딩·DB 인프라 셋업"도 참고하시면 됩니다.

**구성:** 각 `Session`은 가이드(텍스트) → 코드 → 출력 해석(텍스트) 순서로 이어집니다.
**필요:** 전체 인프라(Postgres+pgvector·TEI 임베딩·graph 테이블)가 필요합니다. 인프라가 없으면 `[SKIP]` 경로로 흐름만 검토할 수 있습니다.
**산출물:** `_out/index_stats.json`, `_out/strategy_compare.json`


In [ ]:
import json
import sys
from pathlib import Path

import logging
import warnings
# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력을 읽기 쉽게 라이브러리 로그를 줄인다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치가 필요하다: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
import exaone.integrations.embedding
import exaone.integrations.postgres

TRACK04 = ROOT / "recipes" / "track04_rag_and_knowledge"
DATA = TRACK04 / "data"
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)
print("exaone", exaone.__version__)


**출력 해석:** `exaone <버전>` 한 줄이 출력되면 facade 패턴(`import exaone`) 진입, `load_project_env()` 실행, 통합 서브모듈(`exaone.integrations.embedding`/`postgres`) import가 모두 성공한 것입니다.
- 이후 셀에서 사용할 `ROOT`·`DATA`·`out_dir` 경로와 Postgres/임베딩 헬퍼가 준비됩니다.
- 04b는 LLM 호출을 사용하지 않으므로 EXAONE API 키가 없어도 이 셀은 정상 실행됩니다. 다음 인프라 점검 셀로 진행하세요.


## Session 1. pgvector 셋업 & 인덱싱 점검

먼저 Postgres와 임베딩 서버가 응답하는지, 그리고 vector 테이블이 만들어졌는지 확인합니다. 운영 RAG는 *데이터가 실제로 적재됐는지*부터 검증해야 합니다.


### Session 1-1. Postgres · Embedding 사전 점검

**하는 일:** Postgres·임베딩 서버 가용성과 vector/graph 테이블 존재 여부를 한 번에 점검합니다.

**정상:** `postgres_available`·`embedding_server_reachable`·`graph_tables_available`가 `True`이고, `resolved vector table`이 `data_embeddings`입니다.

**의미:** 운영 RAG는 "데이터와 서비스가 실제로 준비됐는가"부터 확인합니다. 빠진 항목이 있으면 셀이 필요한 셋업 step(1~4)을 안내합니다.


In [ ]:
url = exaone.integrations.postgres.postgres_url_from_env()
emb_url = exaone.integrations.embedding.embedding_base_url_from_env()
PG_OK = exaone.integrations.postgres.postgres_available(url)
EMB_OK = exaone.integrations.embedding.embedding_server_reachable(emb_url)
env_table = exaone.integrations.postgres.vector_table_name_from_env()
resolved_table = exaone.integrations.postgres.resolve_vector_table_name(url, env_table) if PG_OK else None
GRAPH_OK = exaone.integrations.postgres.graph_tables_available(url) if PG_OK else False

print("postgres_available:", PG_OK)
print("embedding_server_reachable:", EMB_OK)
print("PGVECTOR_TABLE_NAME (env):", env_table)
print("resolved vector table:", resolved_table)
print("graph_tables_available:", GRAPH_OK)

if not PG_OK:
    print("\n안내: cd infrastructure/setup && ./step1_downloads.sh && ./step2_docker.sh")
if PG_OK and resolved_table is None:
    print("안내: ./step3_build_rag.sh로 vector 테이블을 생성하세요.")
if PG_OK and not GRAPH_OK:
    print("안내: ./step4_build_graph.sh로 graph 테이블을 생성하세요.")


**출력 해석:** 다섯 가지 인프라 신호가 한 번에 준비됐는지 확인합니다.
- `postgres_available: True` · `embedding_server_reachable: True`이면 DB와 임베딩 서버가 모두 응답한다는 뜻입니다.
- `PGVECTOR_TABLE_NAME (env): data_embeddings`와 `resolved vector table: data_embeddings`가 일치하면 `.env`가 가리키는 테이블이 실제로 존재합니다. `data_embeddings`는 LlamaIndex의 기본 테이블 이름입니다.
- `graph_tables_available: True`이면 graph 전략까지 사용할 수 있습니다.
- 하나라도 False/None이면 아래 `안내:`가 필요한 step(1~4)을 알려줍니다. 인프라 없이 검토 중이라면 이후 셀이 `[SKIP]`만 출력하는 것이 정상입니다.


### Session 1-2. Vector 테이블 통계 — 행 수·저장 차원

**하는 일:** vector 테이블의 행 수와 저장된 임베딩 차원을 읽어 `.env` 설정값과 비교합니다.

**정상:** `row_count`(예: 5000), `stored_embedding_dim` == `env PGVECTOR_EMBEDDING_DIM`(384), `dim_match: True`가 출력됩니다.

**의미:** 저장 차원이 `.env`의 `PGVECTOR_EMBEDDING_DIM`과 어긋나면 검색이 겉으로는 실행되지만 잘못된 결과를 낼 수 있습니다. 이 흔한 문제를 `dim_match`로 확인합니다.


In [ ]:
index_stats = {"postgres_ok": PG_OK, "embedding_ok": EMB_OK, "vector_table": resolved_table, "graph_ok": GRAPH_OK}
if PG_OK and resolved_table:
    import psycopg
    from psycopg import sql as psql

    embed_dim = exaone.integrations.embedding.embedding_dim_from_env()
    with psycopg.connect(url) as conn, conn.cursor() as cur:
        cur.execute(psql.SQL("SELECT COUNT(*) FROM {tbl}").format(tbl=psql.Identifier(resolved_table)))
        row_count = cur.fetchone()[0]
        cur.execute(psql.SQL("SELECT vector_dims(embedding) FROM {tbl} WHERE embedding IS NOT NULL LIMIT 1").format(tbl=psql.Identifier(resolved_table)))
        dim_row = cur.fetchone()
        stored_dim = dim_row[0] if dim_row else None
    index_stats.update({
        "row_count": row_count,
        "stored_embedding_dim": stored_dim,
        "env_embedding_dim": embed_dim,
        "dim_match": stored_dim == embed_dim if stored_dim else None,
    })
    print("row_count:", row_count)
    print("stored_embedding_dim:", stored_dim, "| env PGVECTOR_EMBEDDING_DIM:", embed_dim)
    print("dim_match:", index_stats["dim_match"])
else:
    # (en) Without Postgres + a vector table there are no index stats to read.
    # (kr) Postgres와 vector 테이블이 없으면 읽을 인덱스 통계가 없다.
    print("[SKIP] Postgres·vector 테이블이 연결되지 않았습니다 — 인덱스 통계는 infrastructure/setup(step1~3)을 완료하면 출력됩니다.")


**출력 해석:** vector 테이블이 *적재됐고 차원이 맞는지* 확인합니다.
- `row_count: 5000`은 step3의 `STEP3_MAX_CONTEXTS=5000` 설정만큼 passage(문서 조각)가 들어왔다는 뜻입니다. 값을 0으로 두거나 변수를 제거하면 전체를 적재합니다.
- `stored_embedding_dim: 384 | env PGVECTOR_EMBEDDING_DIM: 384` — 저장 벡터 차원이 `multilingual-e5-small`(384)과 일치합니다.
- `dim_match: True`이면 **흔한 함정**(저장 차원 ≠ env 차원일 때 검색이 조용히 깨지는 문제)을 피했다는 뜻입니다. False이면 데이터를 다시 적재해야 합니다.
- Postgres나 vector 테이블이 없으면 `[SKIP]`만 출력됩니다. 인프라 없이 검토할 때는 정상 동작입니다.


### Session 1-3. 샘플 검색 점검 — vector adapter

**하는 일:** `build_vector_adapter_from_env` 어댑터를 `VectorRetrievalStrategy`에 연결해 한 질의("What is chromatin?")를 검색합니다.

**정상:** chromatin 관련 청크 3건이 코사인 점수(약 0.88~0.89)와 함께 출력됩니다.

**의미:** 인덱스를 실제로 읽을 수 있고 의미 검색이 동작하는지 확인하는 최소 점검입니다.


In [ ]:
smoke_hits = []
if PG_OK and resolved_table and EMB_OK:
    embedder = exaone.integrations.embedding.build_embedder_from_env()
    adapter = exaone.integrations.postgres.build_vector_adapter_from_env(url, embedder, table_name=resolved_table)
    strategy = exaone.retrieval.VectorRetrievalStrategy(
        embed_fn=adapter.embed_fn_for_exaone(),
        search_fn=adapter.search_fn_for_exaone(3),
    )
    chunks = strategy.retrieve("What is chromatin?", top_k=3)
    smoke_hits = [c.to_dict() for c in chunks]
    for i, ch in enumerate(chunks, 1):
        print(f"{i}. score={ch.score:.3f} text={(ch.text or '')[:80].replace(chr(10), ' ')!r}")
else:
    # (en) The smoke search needs Postgres, the vector table, and the embedding server.
    # (kr) 샘플 검색 점검은 Postgres·vector 테이블·임베딩 서버가 모두 있어야 한다.
    print("[SKIP] Postgres·임베딩·vector 테이블이 준비되지 않았습니다 — 샘플 검색 점검은 인프라가 준비되면 실행됩니다.")


**출력 해석:** 어댑터와 `VectorRetrievalStrategy`를 연결해 "What is chromatin?" 한 건을 검색한 간단 점검입니다.
- 상위 3건이 모두 chromatin 관련 결과(`score=0.892 / 0.891 / 0.883`)라면 질의 의미에 맞는 passage(문서 조각)가 올라온 것입니다.
- e5 계열 임베딩의 코사인 점수는 기준선이 높은 편이라 0.88~0.89처럼 좁은 범위에 모일 수 있습니다. 절대값만 보지 말고 *순위와 내용*을 함께 확인해야 합니다. 무관 질의는 0.81 안팎으로 떨어지는 모습을 Session 2에서 비교합니다.
- Postgres·임베딩 서버·vector 테이블이 없으면 `[SKIP]`만 출력됩니다. 인프라 없이 검토할 때는 정상 동작입니다.


### Session 1-4. 산출물 — `index_stats.json`

**하는 일:** 앞 단계 결과(가용성·`row_count`·`dim_match`·샘플 검색 건수)를 `index_stats.json`에 저장합니다.

**정상:** `saved:`와 저장 경로가 출력됩니다.

**의미:** 이 파일은 다음 Session이나 회귀 점검의 입력으로 사용할 수 있습니다.


In [ ]:
from datetime import datetime, timezone

index_stats["generated_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
index_stats["smoke_hit_count"] = len(smoke_hits)
path = out_dir / "index_stats.json"
path.write_text(json.dumps(index_stats, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path.resolve())


**출력 해석:** `saved: …/_out/index_stats.json` 경로가 출력되면 Session 1 점검 결과(가용성·`row_count`·`dim_match`·샘플 검색 건수)가 파일로 저장된 것입니다.
- 이 파일은 다음 Session이나 회귀 점검의 입력으로 사용할 수 있습니다. 인프라가 준비되지 않았다면 `postgres_ok: false`처럼 현재 상태가 그대로 저장됩니다.


## Session 2. 검색 전략 비교 — vector / graph / hybrid

세 전략은 "관련 문서"를 찾는 방식이 다르고, **질의 유형마다 강한 전략도 다릅니다.**
- **vector** — 임베딩 코사인 유사도 기반 검색입니다. *의미·사실 질의*에 강합니다.
- **graph** — 질의의 엔티티·키워드로 그래프를 조회합니다. *명명 엔티티·관계 질의*에 강합니다.
- **hybrid** — vector와 graph 결과를 RRF로 순위 결합합니다(`vector_weight=0.5, graph_weight=0.5, rrf_k=60`). 두 유형을 함께 다룰 때 안정적인 기본값입니다.

**테스트 시나리오** — 적재된 영어 MS MARCO 코퍼스에 맞춰 질의 10건을 **두 비교 유형 + 무관 대조군(q10)**으로 구성했습니다.
- **의미 질의(q01~q06):** 염색질·템스강·integrity·radon·반도체·비글 가격처럼 의미 검색이 중요한 질문입니다. vector가 강할 것으로 기대합니다.
- **엔티티/관계 질의(q07~q09):** Ronald Reagan·Reserve Bank of Australia·Securency처럼 그래프에 있는 명명 엔티티를 직접 부르는 질문입니다. graph가 강할 것으로 기대합니다.
- **무관 질의(q10):** 어떤 전략도 관련 결과를 내지 않는 것이 정상인 대조군입니다. 임계값을 걸었을 때 검색 결과 없음으로 판정되어야 합니다.

**두 가지 함정도 함께 확인합니다:** ① 임계값(`MIN_SCORE`)이 없으면 모든 전략이 항상 `top_k`개를 반환합니다. 그래서 `vec_rel`(임계값 통과 수)로 실제 관련도와 검색 결과 없음 여부를 따로 봐야 합니다. ② 점수 척도(vector 코사인·graph 키워드·hybrid RRF)가 서로 달라 절대값을 직접 비교하면 안 됩니다.


### Session 2-1. 전략 연결

**하는 일:** vector·graph·hybrid 세 전략을 같은 테이블 위에 구성합니다. 인프라가 준비됐을 때만 실행됩니다.

**정상:** `strategies ready: data_embeddings`가 출력됩니다.

**의미:** 인프라가 준비되지 않으면 전략을 `None`으로 두고, 아래 비교 셀이 *모든 칸을 `[SKIP]`*로 남깁니다. 이는 실제 검색 결과 없음과 구분하기 위한 표시입니다.


In [ ]:
# (en) Vector cosine relevance threshold, shared by the Session 2 examples and chart.
# (kr) vector 코사인 관련도 임계값(주제에 맞는 결과를 e5의 ~0.81 기준선에서 분리). Session 2 예제·차트 공용.
MIN_SCORE = 0.84
INFRA_OK = bool(PG_OK and EMB_OK and resolved_table and GRAPH_OK)
vector_strategy = graph_strategy = hybrid_strategy = None

if INFRA_OK:
    embedder = exaone.integrations.embedding.build_embedder_from_env()
    vector_adapter = exaone.integrations.postgres.build_vector_adapter_from_env(url, embedder, table_name=resolved_table)
    graph_adapter = exaone.integrations.postgres.build_graph_adapter_from_env(url)
    vector_strategy = exaone.retrieval.VectorRetrievalStrategy(
        embed_fn=vector_adapter.embed_fn_for_exaone(), search_fn=vector_adapter.search_fn_for_exaone(5))
    graph_strategy = exaone.retrieval.GraphRetrievalStrategy(query_fn=graph_adapter.query_fn_for_exaone(5))
    hybrid_strategy = exaone.retrieval.HybridRetrievalStrategy(
        vector_strategy=vector_strategy, graph_strategy=graph_strategy, vector_weight=0.5, graph_weight=0.5)
    print("strategies ready:", resolved_table)
else:
    # (en) Full infra (Postgres + embedding + vector + graph) is required to wire strategies.
    # (kr) 전략 연결에는 전체 인프라(Postgres+임베딩+vector+graph)가 필요하다.
    print("[SKIP] 전체 인프라(Postgres+임베딩+vector+graph)가 준비되지 않았습니다 — vector/graph/hybrid 전략은 INFRA_OK일 때 구성됩니다.")


**출력 해석:** `strategies ready: data_embeddings`가 보이면 vector·graph·hybrid 세 전략이 같은 테이블 위에 구성된 것입니다.
- 이후 비교 셀이 이 세 전략을 같은 10개 질의로 실행합니다.
- 전체 인프라(Postgres·임베딩·vector/graph 테이블)가 없으면 `[SKIP]`만 출력되고, 비교 표의 모든 칸도 `[SKIP]`로 표시됩니다. 이는 실제 검색 결과 없음과 구분하기 위한 상태입니다.


### Session 2-2. 질의 10건 — 유형별 테스트셋

**하는 일:** `query_fixtures` 10건을 읽어 **유형별(의미 / 엔티티·관계 / 무관)로 전부** 출력합니다.

**입력:** `data/retrieval_query_fixtures.json`

**정상:** 의미 6건 + 엔티티 3건 + 무관 1건 = `queries: 10`.

**의미:** 이 10건이 이후 예제·차트·0-hit 의 공통 테스트셋입니다.


In [ ]:
fixtures = json.loads((DATA / "retrieval_query_fixtures.json").read_text(encoding="utf-8"))
# (en) Group the queries by type so the learner sees the whole test set at a glance.
# (kr) 질의를 유형별로 묶어 전체 테스트셋을 한눈에 보여준다.
by_group = {}
for it in fixtures:
    by_group.setdefault(it.get("group", ""), []).append(it)
labels = {"semantic": "의미 질의 — vector 강세 기대",
          "entity": "엔티티/관계 질의 — graph 강세 기대",
          "0-hit": "무관 질의 — 0-hit 대조군"}
for g in ["semantic", "entity", "0-hit"]:
    print(f"[{labels[g]}]")
    for it in by_group.get(g, []):
        print(f"  {it['id']}  {it['query']}")
print("queries:", len(fixtures))


**출력 해석:** 세 유형으로 묶인 10건이 이번 비교의 테스트셋입니다.
- **의미 질의(q01~q06):** 코퍼스에 답 문장이 있는 사실형 질문 → vector(임베딩 유사도)에 유리할 것으로 기대.
- **엔티티/관계 질의(q07~q09):** 그래프에 있는 명명 엔티티(Reagan·RBA·Securency)를 직접 부름 → graph 에 유리할 것으로 기대.
- **무관 질의(q10):** 코퍼스에 답이 없음 → 어떤 전략도 관련 결과가 0 이어야 정상.
다음 셀부터 *실제 검색 결과* 로 이 기대가 맞는지 확인합니다.


### Session 2-3. 예제로 보는 전략 차이 — 실제 검색 결과

**하는 일:** 유형별 대표 질의(q01 의미 · q07·q09 엔티티)를 세 전략으로 검색해 **각 전략이 실제로 돌려준 top 청크 텍스트** 를 봅니다.

**정상:** vector 는 의미 질의에 맞는 문장, graph 는 엔티티 질의에 맞는 `Entity:` 행, hybrid 는 둘을 병합한 목록.

**의미:** 점수·vec_rel 뒤의 *실제 검색 결과* 를 눈으로 확인해, 다음 차트의 집계가 무슨 뜻인지 체감합니다.


In [ ]:
# (en) Concrete examples: the actual top chunk each strategy returns, to ground the aggregate chart.
# (kr) 구체 예제: 각 전략이 실제로 돌려주는 top 청크 — 집계 차트를 체감하기 위함.
def _mark(name, score):
    if name == "vector":
        return "[O]" if score >= MIN_SCORE else "[X]"
    if name == "graph":
        return "[O]" if score >= 0.85 else "[X]"
    return "   "


def show_example(qid):
    item = next(it for it in fixtures if it["id"] == qid)
    print(f"\n== {qid} [{item.get('group','')}]: {item['query']!r}")
    for name, strat in [("vector", vector_strategy), ("graph", graph_strategy)]:
        top = strat.retrieve(item["query"], top_k=3)[0]
        text = (top.text or "").replace(chr(10), " ")[:78]
        print(f"   {name:<7} {top.score:.3f} {_mark(name, top.score)}  {text!r}")
    # (en) Hybrid merges vector+graph (RRF); show top-2 with source tags so the blend is visible.
    # (kr) hybrid 는 vector+graph 를 RRF 로 병합 — 출처 태그와 함께 상위 2건을 보여 병합을 드러낸다.
    print("   hybrid  (vector+graph 병합 상위 2건):")
    for x in hybrid_strategy.retrieve(item["query"], top_k=3)[:2]:
        src = (x.metadata or {}).get("_source", "?")
        print(f"           [{src:<6}] {(x.text or '').replace(chr(10), ' ')[:64]!r}")


if vector_strategy is not None:
    # (en) One per type: q01 semantic (vector wins), q07/q09 entity (graph wins; q09 vector misses).
    # (kr) 유형별: q01 의미(vector 승)·q07·q09 엔티티(graph 승; q09 는 vector 헛다리).
    for qid in ["q01", "q07", "q09"]:
        show_example(qid)
else:
    print("[SKIP] 전략 미구성(INFRA_OK=False) — 예제 비교는 인프라 준비 시 실행됩니다.")


**출력 해석:** 세 예제가 "어떤 질의가 어떤 전략에 맞는지" 를 실제 결과로 보여줍니다.
- **q01 "What is chromatin?" (의미):** vector `0.892 [O]` 가 *"Chromatin. The nucleus contains the chromosomes…"* 정답 문장을 반환합니다. graph 는 `0.300 [X]` 로 *"Entity: Reserve Bank of Australia…"* — 질의에 엔티티 이름이 없어 무관 hub 를 돌려줍니다. hybrid 상위는 vector 의 chromatin 문장입니다.
- **q07 "Who was Ronald Reagan?" (엔티티):** 둘 다 적중 — vector 는 *"Ronald Wilson Reagan (1911–2004)…"* 전기 문장, graph 는 `0.950 [O]` 로 *"Entity: Ronald Reagan (PER) … Democratic Party"* 엔티티 + 연결을 반환합니다. hybrid 목록엔 **vector 문장 + graph 엔티티** 가 함께 들어옵니다.
- **q09 "What is Securency?" (엔티티):** **vector 가 헛다리** — `0.813 [X]` 로 엉뚱한 *"Integrity. Integrity is a concept…"* 를 반환(임계값 미달, vec_rel=0). graph 는 `0.950 [O]` 로 *"Entity: Securency … RBA reputation affected"* 를 정확히 찾습니다. hybrid 는 top1 이 vector 의 오답이지만 **2위에 graph 의 Securency** 가 들어와 누락을 복구합니다(단 RRF 동점이라 순위 보장은 아님).
- 정리: **의미 질의→vector · 엔티티 질의→graph 가 적중하고, hybrid 는 둘을 합쳐 누락을 줄입니다.**


### Session 2-4. 전체 비교 — 전략×질의 관련성 매트릭스

**하는 일:** 2-3 예제의 패턴이 10질의 전반에서 성립하는지, **전략(행)×질의(열) 히트맵** 으로 "관련 top 결과를 냈나(O)" 를 한눈에 봅니다.

**정상:** vector 가 의미 질의를 넓게 커버 · graph 는 엔티티(q07~q09)에서만 O · hybrid 는 둘의 합집합(q09 까지 커버, q10 만 전무). (matplotlib 없으면 텍스트 매트릭스, 인프라 없으면 `[SKIP]`)

**의미:** "어느 질의에서 어느 전략이 이기나" 와 hybrid 가 왜 robust 한지(=합집합)를 한 장으로 봅니다.


In [ ]:
# (en) Summarize one strategy across all fixtures; None strategy yields skipped rows.
# (kr) 한 전략을 전체 fixture 로 요약한다. 전략이 None 이면 skipped 행을 만든다.
def summarize_strategy(name, strategy):
    rows = []
    if strategy is None:
        for item in fixtures:
            rows.append({"query_id": item["id"], "group": item.get("group", ""), "strategy": name, "hit_count": 0, "top_score": 0.0, "relevant": None, "skipped": True})
        return rows
    for item in fixtures:
        chunks = strategy.retrieve(item["query"], top_k=item.get("top_k", 5))
        top_score = float(chunks[0].score) if chunks else 0.0
        # (en) 'relevant' = hits clearing MIN_SCORE; only meaningful for vector cosine.
        # (kr) 'relevant' = MIN_SCORE 를 넘는 hit 수; vector 코사인에만 의미.
        relevant = sum(1 for c in chunks if (c.score or 0.0) >= MIN_SCORE) if name == "vector" else None
        rows.append({"query_id": item["id"], "group": item.get("group", ""), "strategy": name, "hit_count": len(chunks), "top_score": round(top_score, 4), "relevant": relevant, "skipped": False})
    return rows


rows = []
rows.extend(summarize_strategy("vector", vector_strategy))
rows.extend(summarize_strategy("graph", graph_strategy))
rows.extend(summarize_strategy("hybrid", hybrid_strategy))

# (en) Per-query "did this strategy surface a relevant result?": vector = any cosine >= MIN_SCORE; graph = exact-entity match (top1 >= GRAPH_REL); hybrid = union of the two (RRF merges both).
# (kr) 질의별 "이 전략이 관련 결과를 냈나": vector = 코사인 >= MIN_SCORE 1건+; graph = 정확 엔티티 매칭(top1 >= GRAPH_REL); hybrid = 둘의 합집합(RRF 가 둘을 병합).
GRAPH_REL = 0.85
qids = [it["id"] for it in fixtures]
vrow = {r["query_id"]: r for r in rows if r["strategy"] == "vector"}
grow = {r["query_id"]: r for r in rows if r["strategy"] == "graph"}
vector_rel = {q: (not vrow[q]["skipped"]) and ((vrow[q]["relevant"] or 0) >= 1) for q in qids}
graph_rel = {q: (not grow[q]["skipped"]) and (grow[q]["top_score"] >= GRAPH_REL) for q in qids}
hybrid_rel = {q: (vector_rel[q] or graph_rel[q]) for q in qids}
grid = [[1 if vector_rel[q] else 0 for q in qids],
        [1 if graph_rel[q] else 0 for q in qids],
        [1 if hybrid_rel[q] else 0 for q in qids]]

any_real = any(not r["skipped"] for r in rows)
try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

if any_real and HAS_MPL:
    strat_names = ["vector", "graph", "hybrid"]
    groups = [it.get("group", "") for it in fixtures]
    gcolor = {"semantic": "#2e86c1", "entity": "#1e8449", "0-hit": "#c0392b"}
    fig, ax = plt.subplots(figsize=(11, 2.7))
    ax.imshow(grid, cmap=ListedColormap(["#ecf0f1", "#27ae60"]), vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(qids)))
    ax.set_xticklabels(qids)
    ax.set_yticks(range(len(strat_names)))
    ax.set_yticklabels(strat_names)
    # (en) Color query labels by group; separate the three groups with vertical lines.
    # (kr) 질의 라벨을 유형별 색으로; 세 유형 구간을 세로선으로 구분.
    for tick, g in zip(ax.get_xticklabels(), groups):
        tick.set_color(gcolor.get(g, "#000000"))
    for x in (5.5, 8.5):
        ax.axvline(x, color="#34495e", lw=1.2)
    # (en) Mark relevant cells with O.
    # (kr) 관련 셀에 O 표시.
    for i in range(len(strat_names)):
        for j in range(len(qids)):
            if grid[i][j]:
                ax.text(j, i, "O", ha="center", va="center", color="white", fontweight="bold")
    ax.set_title("Which strategy returns a relevant result?   (O = relevant top-k hit; blue=semantic, green=entity, red=0-hit)")
    fig.tight_layout()
    plt.show()
    print("vector relevant:", [q for q in qids if vector_rel[q]])
    print("graph  relevant:", [q for q in qids if graph_rel[q]], "  <- entity queries only")
    print("hybrid relevant:", [q for q in qids if hybrid_rel[q]], "  <- union: recovers q09 (vector misses); only q10 fails")
elif any_real:
    # (en) matplotlib absent: text matrix fallback.
    # (kr) matplotlib 미설치: 텍스트 매트릭스로 대체.
    print("[SKIP] matplotlib 미설치 — 히트맵 생략(pip install matplotlib 후 재실행). 텍스트 매트릭스:")
    print("         " + "  ".join(qids))
    for nm, relmap in [("vector", vector_rel), ("graph", graph_rel), ("hybrid", hybrid_rel)]:
        print(f"  {nm:<7} " + "    ".join("O" if relmap[q] else "." for q in qids))
else:
    # (en) No wired strategies (offline) — nothing to plot.
    # (kr) 전략 미배선(오프라인) — 그릴 데이터가 없다.
    print("[SKIP] 풀 인프라 미충족 — 비교 히트맵은 INFRA_OK 일 때 출력됩니다.")


**출력 해석:** 매트릭스 한 칸 = "그 전략이 그 질의에 *관련 top 결과* 를 냈나"(O·초록 = 관련).
- **vector(1행):** q01~q08 을 넓게 커버하지만 **q09(Securency)·q10 은 실패** — 의미·일반 사실엔 강하나 코퍼스에 드문 niche 엔티티는 놓칩니다.
- **graph(2행):** **엔티티 질의 q07~q09 에서만 O** — 질의가 그래프의 명명 엔티티를 정확히 부를 때만 작동(나머지는 무관 hub 라 ✗).
- **hybrid(3행):** vector·graph 의 **합집합** — q01~q09 전부 커버합니다. 특히 **q09 는 vector 가 놓치지만 graph 덕에 hybrid 가 건집니다.** 오직 q10(무관)만 전 전략 실패 → 진짜 0-hit.
- 한 줄 결론: **의미→vector · 엔티티→graph · hybrid=둘의 합집합(robust).** graph 의 단독 기여는 q09 한 곳뿐이지만, 바로 그것이 hybrid 를 쓰는 이유입니다. (관련 판정: vector 코사인 ≥ MIN_SCORE · graph top1 ≥ 0.85 · hybrid = 합집합)


### Session 2-5. 검색 결과 없음 케이스 — 임계값이 없으면 0건 판정도 없다

**하는 일:** 무관 질의 q10을 세 전략으로 실행해 결과 수(`hits`), 최고 점수(`top_score`), vector의 임계값 통과 수(`vec_rel`)를 확인합니다.

**정상:** 세 전략 모두 `hits=top_k`(5)를 반환하지만, vector의 `vec_rel(>=MIN_SCORE)=0`입니다. 임계값을 걸어야 실제 검색 결과 없음으로 판정할 수 있습니다.

**의미:** "무관한 질의 = 자동으로 검색 결과 0건"이 **아닙니다.** 최근접/키워드 검색은 임계값이 없으면 무엇이든 '가장 가까운' `top_k`개를 반환합니다. 운영에서는 **저품질 컨텍스트가 조용히 섞이는** 실패 모드가 될 수 있습니다(04a Session 3-2 "빈 검색은 예외가 아니라 빈 결과로"와 연결). min-score 임계값이나 재랭킹으로 걸러야 합니다.


In [ ]:
zero_hit_query = fixtures[-1]["query"]
zero_row = {"query": zero_hit_query, "results": {}}
if hybrid_strategy is not None:
    print(f"검색한 질의: {zero_hit_query!r}\n")
    # (en) Show the actual top chunk each strategy returns for the nonsense query.
    # (kr) 무의미 질의에 각 전략이 실제로 돌려준 top 청크를 보여준다.
    for name, strat in [("vector", vector_strategy), ("graph", graph_strategy), ("hybrid", hybrid_strategy)]:
        chunks = strat.retrieve(zero_hit_query, top_k=5)
        top = chunks[0]
        vec_rel = sum(1 for c in chunks if (c.score or 0.0) >= MIN_SCORE) if name == "vector" else None
        zero_row["results"][name] = {"hits": len(chunks), "top_score": round(float(top.score), 4), "relevant_vec": vec_rel}
        extra = f"  vec_rel(>={MIN_SCORE})={vec_rel}" if vec_rel is not None else ""
        print(f"{name:<7} hits={len(chunks)} top={top.score:.3f}{extra}")
        print(f"        -> {(top.text or '').replace(chr(10), ' ')[:72]!r}")
    print("\n-> 돌아온 건 'foul weather gear...'/'Slime mold...' 처럼 질의와 무관한 baseline 청크입니다.")
    print("   vector 에 MIN_SCORE 를 걸면 vec_rel=0 -> 진짜 0-hit.")
else:
    # (en) No strategies were wired, so the zero-hit comparison cannot run.
    # (kr) 전략이 구성되지 않아 0-hit 비교를 실행할 수 없다.
    print("[SKIP] 전략 미구성(INFRA_OK=False) — 0-hit 비교는 인프라 준비 시 실행됩니다.")


**출력 해석:** 무관 질의 *"zzzzzzzz … 99999"* 에 각 전략이 돌려준 실제 결과입니다.
- 셋 다 `hits=5`(=top_k) — 임계값이 없으면 무의미 질의에도 '가장 가까운' 청크가 채워집니다.
- 돌아온 내용은 *"Most foul weather gear…"*(vector)·*"Entity: Slime (PER) Slime mold…"*(graph)처럼 **질의와 완전히 무관** 하고, vector 점수도 `≈0.814` 로 baseline 바닥입니다.
- vector `vec_rel(>=0.84)=0` — **임계값을 걸어야 비로소 "결과 없음(0-hit)"** 으로 처리됩니다. 운영 RAG 가 무관 질의에 헛컨텍스트를 답에 섞지 않으려면 이 컷이 필요합니다(04a Session 3-2 와 연결).
- 풀 인프라가 없으면 `[SKIP]` 만 출력됩니다 — 인프라 없이 검토할 때는 정상입니다.


### Session 2-6. 산출물 — `strategy_compare.json`

**하는 일:** 비교 결과(30행 + 검색 결과 없음 케이스)를 `strategy_compare.json`에 저장합니다.

**정상:** `saved:`와 저장 경로가 출력됩니다.

**의미:** 각 행에 `relevant`(`vec_rel`)·`skipped`가 포함되므로 회귀 점검의 입력으로 사용할 수 있습니다.


In [ ]:
payload = {
"generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
"infra_ok": INFRA_OK,
"rows": rows,
"zero_hit_case": zero_row,
}
path = out_dir / "strategy_compare.json"
path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path.resolve())


**출력 해석:** `saved: …/_out/strategy_compare.json`이 출력되면 30행(전략×질의)과 검색 결과 없음 케이스가 파일로 저장된 것입니다.
- 각 행에 `relevant`(vector의 `vec_rel`)·`skipped`가 포함되어, 회귀 점검에서 임계값 통과 수와 인프라 여부를 그대로 비교할 수 있습니다.


## 체크포인트

- [ ] `index_stats.json` — Postgres가 준비되어 있으면 `dim_match: true`입니다(저장 차원 == env 차원).
- [ ] `strategy_compare.json` — 인프라가 준비되어 있으면 vector/graph/hybrid 각각 10개 질의 행과 검색 결과 없음 케이스가 포함됩니다.
- [ ] 인프라가 없어도 두 산출물이 *skipped 표시와 함께* 저장됩니다.
